# 审查者Agent：将构建与标记分开

写代码的agent不能为其打分。一个审查者应该是第二个循环，有不同的系统提示词、目标、以及对构建者产出的任何物料只读的权限。构建者和审查者之间的差距是可靠性的基石。

## 问题描述

你让agent修复一个bug，它编辑了四个文件、跑了测试、然后报告完成了。验证门禁确认验收，且没有超过范围。两天之后你发现只修复了bug错误的一半。

验收很必要、但不够。审查者问的是验收范围外的问题：是否解决了正确的问题？有没有不动声色地扩大范围？本应该被质疑的假设是否归档？工作台的状态是否支持下一个会话顺利接手？

## 基本概念
```mermaid
flowchart TD
A[Builder Agent] --> B[diff + state + feedback + verdict] --> C[Reviewer Agent] --> D[reviewer_checklist.md]
C --> E[review_report.json] --> F[Human Sign-off]
```

### 审查规则

|维度|问题|
|---|---|
|问题契合|是否真的解决了强调的问题，而不是一个近似问题|
|范围记录|编辑内容是否遵循了约束，约束是否被故意放大了|
|假设|是否所有的隐式假设都被记录并等待审查|
|验证质量|验收命令真实证明了目标，而非目标的弱形式|
|交接就绪|下一个会话是否能从当前状态顺利启动|

每个维度0-2分，小于7分视为软失败，小于5分视为硬失败。

### 审查者是独立的角色，不是独立的模型

可以为审查者使用与构建者相同的模型。区别是角色区分：不同的是系统提示词，不同的输入，没有对差异的修改权限。姿态的概念就是信号的改变。

### 审查者不能编辑差异

审查者读差异、状态、反馈、验证。然后写一份报告。不会为差异打补丁。如果报告说需要修复，下一轮构建者会对此负责，然后审查者接着进行审查。不要混淆角色。

### 审查规则和验证门禁

验证门禁检查的是确定性事实：验收跑了没有、规则遵守了没有、修改是否在范围内。
审查做的是定性判断：是不是正确的活儿、有没有记录、交接能不能用。
两者都需要。

# 开始编码

对应本章核心：**构建 / 审查角色分离（同模型不同提示词与输入）**、**五维 0–2 分（<7 软失败，<5 硬失败）**、**审查者只读差异并写 `review_report.json`**、**门禁≠审查**。  
玩具用启发式打分保证可断言；生产段用 **同一 DeepSeek + 只读工具** 做定性审查。不硬凑 PyTorch。


## 1. 教学玩具：ReviewerLoop + 启发式五维

- 物料：goal / diff / state / feedback / verification。
- `patch_diff` 对审查者抛 `RoleError`。
- 清单写入 `reviewer_checklist.md`。


In [ ]:
from __future__ import annotations

import json
import shutil
import tempfile
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Literal


DIMENSIONS = (
    "problem_fit",
    "scope_record",
    "assumptions",
    "verification_quality",
    "handoff_ready",
)

Verdict = Literal["pass", "soft_fail", "hard_fail"]


@dataclass
class DimensionScore:
    """单维 0–2 分。"""

    dimension: str
    score: int
    note: str

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


@dataclass
class ReviewReport:
    """审查报告：定性判断，不改差异。"""

    task_id: str
    scores: list[dict[str, Any]]
    total: int
    verdict: Verdict
    role: str = "reviewer"
    edited_diff: bool = False

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


class RoleError(RuntimeError):
    """审查者越权（试图改差异）。"""


def verdict_from_total(total: int) -> Verdict:
    """
    Args:
        total: 五维总分（0–10）。
    Returns:
        verdict: <5 硬失败，<7 软失败，否则通过。
    """
    if total < 5:
        return "hard_fail"
    if total < 7:
        return "soft_fail"
    return "pass"


def clamp_score(n: int) -> int:
    return max(0, min(2, int(n)))


class ArtifactBundle:
    """构建者产出的只读物料。"""

    def __init__(self, root: Path) -> None:
        self.root = Path(root)

    def _load(self, name: str) -> Any:
        p = self.root / name
        if not p.exists():
            return {} if name.endswith(".json") else []
        if name.endswith(".jsonl"):
            return [json.loads(ln) for ln in p.read_text(encoding="utf-8").splitlines() if ln.strip()]
        return json.loads(p.read_text(encoding="utf-8"))

    def goal(self) -> str:
        c = self._load("scope_contract.json")
        return str(c.get("goal", ""))

    def diff(self) -> dict[str, Any]:
        return self._load("diff.json")

    def state(self) -> dict[str, Any]:
        return self._load("agent_state.json")

    def feedback(self) -> list[dict[str, Any]]:
        rows = self._load("feedback_record.jsonl")
        return rows if isinstance(rows, list) else []

    def verification(self) -> dict[str, Any]:
        return self._load("verification_report.json")

    def snapshot(self) -> dict[str, Any]:
        """
        Returns:
            payload: 审查者可读的全部物料。
        """
        return {
            "goal": self.goal(),
            "diff": self.diff(),
            "state": self.state(),
            "feedback": self.feedback(),
            "verification": self.verification(),
        }


class HeuristicReviewer:
    """玩具审查：按物料启发式打分（可断言）。生产段再用 LLM 做定性。"""

    def score(self, bundle: ArtifactBundle) -> ReviewReport:
        """
        Args:
            bundle: 只读物料。
        Returns:
            report: 五维分 + 裁决。
        """
        snap = bundle.snapshot()
        goal = str(snap["goal"])
        diff = snap["diff"] or {}
        state = snap["state"] or {}
        feedback = snap["feedback"] or []
        gate = snap["verification"] or {}
        changed = list(diff.get("files", []))
        summary = str(diff.get("summary", ""))
        forbidden = list(diff.get("forbidden_hits", []))
        expanded = bool(diff.get("scope_expanded", False))

        scores: list[DimensionScore] = []

        # 问题契合：差异摘要是否碰到 goal 关键词
        g = goal.replace(" ", "").replace("，", "")
        tokens = [g[i : i + 2] for i in range(max(0, len(g) - 1))] if len(g) >= 2 else ([g] if g else [])
        blob = summary + json.dumps(changed, ensure_ascii=False)
        hit = any(t in blob for t in tokens)
        scores.append(
            DimensionScore(
                "problem_fit",
                2 if hit and not diff.get("wrong_problem") else (1 if hit else 0),
                "summary/files touch the stated goal" if hit else "diff does not mention the goal",
            )
        )

        # 范围记录：禁止项 / 静默扩大
        if forbidden or expanded:
            sc, note = 0, f"scope creep: forbidden={forbidden} expanded={expanded}"
        elif changed:
            sc, note = 2, "edits stay inside recorded constraints"
        else:
            sc, note = 1, "no files recorded"
        scores.append(DimensionScore("scope_record", sc, note))

        # 假设：state.assumptions 非空
        assumptions = list(state.get("assumptions") or [])
        scores.append(
            DimensionScore(
                "assumptions",
                2 if assumptions else 0,
                "assumptions archived" if assumptions else "implicit assumptions missing",
            )
        )

        # 验证质量：门禁过 + 验收命令存在
        gate_ok = bool(gate.get("passed"))
        acc_ran = any(r.get("exit_code") == 0 for r in feedback)
        weak = bool(gate.get("weak_proxy"))
        if gate_ok and acc_ran and not weak:
            sc, note = 2, "gate passed and acceptance is real"
        elif gate_ok or acc_ran:
            sc, note = 1, "partial / weak verification"
        else:
            sc, note = 0, "no real verification evidence"
        scores.append(DimensionScore("verification_quality", sc, note))

        # 交接就绪：next_action + 状态完整
        nxt = str(state.get("next_action") or "").strip()
        task = state.get("active_task_id")
        if nxt and task:
            sc, note = 2, "next session can resume from state"
        elif nxt or task:
            sc, note = 1, "handoff incomplete"
        else:
            sc, note = 0, "state not handoff-ready"
        scores.append(DimensionScore("handoff_ready", sc, note))

        total = sum(s.score for s in scores)
        return ReviewReport(
            task_id=str(state.get("active_task_id") or "UNKNOWN"),
            scores=[s.to_dict() for s in scores],
            total=total,
            verdict=verdict_from_total(total),
        )


class ReviewerLoop:
    """第二循环：只读构建物料，只写 review_report.json。"""

    def __init__(self, root: Path) -> None:
        self.root = Path(root)
        self.bundle = ArtifactBundle(self.root)
        self.report_path = self.root / "review_report.json"
        self.checklist_path = self.root / "reviewer_checklist.md"
        self.diff_path = self.root / "diff.json"

    def write_checklist(self) -> Path:
        """
        Returns:
            path: 五维审查清单。
        """
        lines = [
            "# Reviewer checklist",
            "",
            "|维度|问题|分|",
            "|---|---|---|",
            "|问题契合|是否解决了强调的问题，而不是近似问题|0-2|",
            "|范围记录|编辑是否遵循约束，约束是否被故意放大|0-2|",
            "|假设|隐式假设是否已记录并等待审查|0-2|",
            "|验证质量|验收是否真证明目标，而非弱形式|0-2|",
            "|交接就绪|下一会话能否从当前状态启动|0-2|",
            "",
            "总分 <7 软失败，<5 硬失败。审查者不得编辑差异。",
            "",
        ]
        self.checklist_path.write_text("\n".join(lines), encoding="utf-8")
        return self.checklist_path

    def review(self, reviewer: HeuristicReviewer | None = None) -> ReviewReport:
        """
        Args:
            reviewer: 打分器，默认启发式。
        Returns:
            report: 已落盘报告。
        """
        self.write_checklist()
        report = (reviewer or HeuristicReviewer()).score(self.bundle)
        self.report_path.write_text(
            json.dumps(report.to_dict(), ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )
        return report

    def patch_diff(self, *_args: Any, **_kwargs: Any) -> None:
        """审查者没有写差异的权限。"""
        raise RoleError("reviewer cannot edit the diff; builder owns the next fix loop")


def write_builder_bundle(
    root: Path,
    *,
    quality: Literal["good", "weak", "bad"] = "good",
) -> Path:
    """
    写入构建者物料。

    Args:
        root: 工作目录。
        quality: good / weak / bad。
    Returns:
        root: 同一目录。
    """
    root.mkdir(parents=True, exist_ok=True)
    goal = "修复登陆校验边界"
    if quality == "good":
        diff = {
            "files": ["src/auth/login.py"],
            "summary": "修复登陆校验边界，拒绝空密码",
            "forbidden_hits": [],
            "scope_expanded": False,
            "wrong_problem": False,
        }
        state = {
            "active_task_id": "T-LOGIN",
            "assumptions": ["登陆接口已有 pytest 骨架"],
            "next_action": "等审查后合并",
            "blockers": [],
        }
        feedback = [{"id": "1", "command": ["pytest"], "exit_code": 0}]
        gate = {"passed": True, "weak_proxy": False}
    elif quality == "weak":
        diff = {
            "files": ["src/auth/login.py"],
            "summary": "触及登陆校验但只改了日志",
            "forbidden_hits": [],
            "scope_expanded": False,
            "wrong_problem": False,
        }
        state = {"active_task_id": "T-LOGIN", "assumptions": [], "next_action": "", "blockers": []}
        feedback = [{"id": "1", "command": ["echo", "ok"], "exit_code": 0}]
        gate = {"passed": True, "weak_proxy": True}
    else:
        diff = {
            "files": ["README.md", "src/email/send.py"],
            "summary": "顺便加了暗主题和邮件重试",
            "forbidden_hits": ["README.md"],
            "scope_expanded": True,
            "wrong_problem": True,
        }
        state = {"active_task_id": None, "assumptions": [], "next_action": "", "blockers": []}
        feedback = []
        gate = {"passed": False, "weak_proxy": True}

    (root / "scope_contract.json").write_text(
        json.dumps({"task_id": "T-LOGIN", "goal": goal}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    (root / "diff.json").write_text(json.dumps(diff, ensure_ascii=False, indent=2), encoding="utf-8")
    (root / "agent_state.json").write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding="utf-8")
    (root / "feedback_record.jsonl").write_text(
        "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in feedback),
        encoding="utf-8",
    )
    (root / "verification_report.json").write_text(
        json.dumps(gate, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return root


print("reviewer ready | five dimensions + read-only second loop")


## 2. 玩具示例

good 通过、weak 软失败、bad 硬失败且差异未被改写。


In [ ]:
def demo_reviewer() -> None:
    """好/弱/坏物料、角色隔离、清单+报告落盘。"""
    root = Path(tempfile.mkdtemp(prefix="reviewer_"))
    try:
        write_builder_bundle(root, quality="good")
        loop = ReviewerLoop(root)
        good = loop.review()
        assert good.total >= 7 and good.verdict == "pass"
        assert loop.checklist_path.exists() and loop.report_path.exists()
        print("good bundle pass ok")

        write_builder_bundle(root, quality="weak")
        weak = ReviewerLoop(root).review()
        assert weak.verdict == "soft_fail"
        assert 5 <= weak.total < 7
        print("weak bundle soft_fail ok")

        write_builder_bundle(root, quality="bad")
        before = (root / "diff.json").read_text(encoding="utf-8")
        bad_loop = ReviewerLoop(root)
        bad = bad_loop.review()
        assert bad.verdict == "hard_fail" and bad.total < 5
        try:
            bad_loop.patch_diff({"files": ["hack.py"]})
            raise AssertionError("reviewer must not patch")
        except RoleError:
            pass
        after = (root / "diff.json").read_text(encoding="utf-8")
        assert after == before
        print("bad bundle hard_fail + no diff edit ok")

        # 门禁过了仍可能审查失败（验收不够）
        assert weak.verdict != "pass"
        print("gate pass != review pass ok")

        print("TOY DEMO OK")
    finally:
        shutil.rmtree(root, ignore_errors=True)


demo_reviewer()


## 3. 生产级：LangChain 审查者工具 + DeepSeek

审查者工具全是读 + `write_review_report`；`try_patch_diff` 演示越权失败。需 `DEEPSEEK_API_KEY`。


In [ ]:
import json
import os
import shutil
import sys
import tempfile
import threading
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"
PROD_ROOT: Path | None = None
PROD_LOOP: ReviewerLoop | None = None
PROD_LOCK = threading.Lock()


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def reset_review_bundle(*, quality: str = "weak") -> str:
    """
    Args:
        quality: good / weak / bad。
    Returns:
        json: root。
    """
    global PROD_ROOT, PROD_LOOP
    if PROD_ROOT and PROD_ROOT.exists():
        shutil.rmtree(PROD_ROOT, ignore_errors=True)
    if quality not in {"good", "weak", "bad"}:
        quality = "weak"
    PROD_ROOT = Path(tempfile.mkdtemp(prefix="reviewer_prod_"))
    write_builder_bundle(PROD_ROOT, quality=quality)  # type: ignore[arg-type]
    PROD_LOOP = ReviewerLoop(PROD_ROOT)
    PROD_LOOP.write_checklist()
    return json.dumps({"root": str(PROD_ROOT), "quality": quality}, ensure_ascii=False)


def _loop() -> ReviewerLoop:
    if PROD_LOOP is None:
        reset_review_bundle(quality="weak")
    assert PROD_LOOP is not None
    return PROD_LOOP



class EmptyArgs(BaseModel):
    pass


class QualityArgs(BaseModel):
    quality: str = Field("weak", description="good / weak / bad")


class ScoreItem(BaseModel):
    dimension: str
    score: int = Field(ge=0, le=2)
    note: str = ""


class ReviewArgs(BaseModel):
    task_id: str = "T-LOGIN"
    problem_fit: int = Field(ge=0, le=2)
    scope_record: int = Field(ge=0, le=2)
    assumptions: int = Field(ge=0, le=2)
    verification_quality: int = Field(ge=0, le=2)
    handoff_ready: int = Field(ge=0, le=2)
    notes_json: str = Field("{}", description="维度->评语 的 JSON 对象")


class PatchArgs(BaseModel):
    summary: str = Field(description="想写入 diff 的摘要（审查者调用应被拒）")


def build_reviewer_tools() -> list[StructuredTool]:
    def _reset(**kwargs: Any) -> str:
        with PROD_LOCK:
            return reset_review_bundle(quality=QualityArgs(**kwargs).quality)

    def _read(name: str) -> str:
        snap = _loop().bundle.snapshot()
        return json.dumps(snap.get(name, snap) if name != "all" else snap, ensure_ascii=False)

    def _goal(**kwargs: Any) -> str:
        with PROD_LOCK:
            return json.dumps({"goal": _loop().bundle.goal()}, ensure_ascii=False)

    def _diff(**kwargs: Any) -> str:
        with PROD_LOCK:
            return json.dumps(_loop().bundle.diff(), ensure_ascii=False)

    def _state(**kwargs: Any) -> str:
        with PROD_LOCK:
            return json.dumps(_loop().bundle.state(), ensure_ascii=False)

    def _feedback(**kwargs: Any) -> str:
        with PROD_LOCK:
            return json.dumps(_loop().bundle.feedback(), ensure_ascii=False)

    def _gate(**kwargs: Any) -> str:
        with PROD_LOCK:
            return json.dumps(_loop().bundle.verification(), ensure_ascii=False)

    def _checklist(**kwargs: Any) -> str:
        with PROD_LOCK:
            p = _loop().write_checklist()
            return p.read_text(encoding="utf-8")

    def _write_review(**kwargs: Any) -> str:
        a = ReviewArgs(**kwargs)
        try:
            notes = json.loads(a.notes_json) if a.notes_json else {}
        except json.JSONDecodeError:
            notes = {}
        dim_scores = {
            "problem_fit": a.problem_fit,
            "scope_record": a.scope_record,
            "assumptions": a.assumptions,
            "verification_quality": a.verification_quality,
            "handoff_ready": a.handoff_ready,
        }
        scores = [
            DimensionScore(d, clamp_score(dim_scores[d]), str(notes.get(d, ""))).to_dict()
            for d in DIMENSIONS
        ]
        total = sum(int(s["score"]) for s in scores)
        report = ReviewReport(
            task_id=a.task_id,
            scores=scores,
            total=total,
            verdict=verdict_from_total(total),
            edited_diff=False,
        )
        with PROD_LOCK:
            _loop().report_path.write_text(
                json.dumps(report.to_dict(), ensure_ascii=False, indent=2) + "\n",
                encoding="utf-8",
            )
            return json.dumps(report.to_dict(), ensure_ascii=False)

    def _try_patch(**kwargs: Any) -> str:
        with PROD_LOCK:
            try:
                _loop().patch_diff(PatchArgs(**kwargs).summary)
                return json.dumps({"ok": True}, ensure_ascii=False)
            except RoleError as e:
                return json.dumps({"ok": False, "error": str(e)}, ensure_ascii=False)

    def _read_report(**kwargs: Any) -> str:
        with PROD_LOCK:
            p = _loop().report_path
            if not p.exists():
                return json.dumps({"ok": False, "error": "review_report.json missing"}, ensure_ascii=False)
            return p.read_text(encoding="utf-8")

    return [
        StructuredTool.from_function(name="reset_bundle", description="重建构建者物料。", func=_reset, args_schema=QualityArgs),
        StructuredTool.from_function(name="read_goal", description="只读目标。", func=_goal, args_schema=EmptyArgs),
        StructuredTool.from_function(name="read_diff", description="只读差异，不能改。", func=_diff, args_schema=EmptyArgs),
        StructuredTool.from_function(name="read_state", description="只读 agent_state。", func=_state, args_schema=EmptyArgs),
        StructuredTool.from_function(name="read_feedback", description="只读反馈日志。", func=_feedback, args_schema=EmptyArgs),
        StructuredTool.from_function(name="read_verification", description="只读验证门禁报告。", func=_gate, args_schema=EmptyArgs),
        StructuredTool.from_function(name="read_checklist", description="读取 reviewer_checklist.md。", func=_checklist, args_schema=EmptyArgs),
        StructuredTool.from_function(
            name="write_review_report",
            description="按五维 0-2 分写入 review_report.json；不修改 diff。",
            func=_write_review,
            args_schema=ReviewArgs,
        ),
        StructuredTool.from_function(
            name="try_patch_diff",
            description="故意暴露的写差异入口；审查者调用应失败。",
            func=_try_patch,
            args_schema=PatchArgs,
        ),
        StructuredTool.from_function(name="read_review_report", description="读取已写报告。", func=_read_report, args_schema=EmptyArgs),
    ]


REVIEWER_TOOLS = build_reviewer_tools()


def build_reviewer_agent():
    """
    Returns:
        agent: 与构建者可同模型，但提示词与工具不同。
    """
    system = (
        "你是审查者，不是构建者。同一模型，不同角色。\n"
        "只读 goal/diff/state/feedback/verification，按五维各打 0-2 分，"
        "然后 write_review_report。不要 try_patch_diff。不要改代码。\n"
        "门禁通过不等于审查通过。用中文短评语。"
    )
    return create_agent(get_llm(), REVIEWER_TOOLS, system_prompt=system)


def _print_trace(messages: list[BaseMessage], limit: int = 30) -> None:
    n = 0
    for m in messages:
        if n >= limit:
            break
        if isinstance(m, HumanMessage):
            print(f"USER: {m.content}")
            n += 1
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    print(f"ACTION: {tc['name']}({tc.get('args', {})})")
                    n += 1
            elif m.content:
                print(f"ASSISTANT: {str(m.content)[:280]}")
                n += 1
        elif isinstance(m, ToolMessage):
            print(f"OBS[{m.name}]: {str(m.content)[:240]}")
            n += 1


print(f"reviewer production ready | {MODEL}")


## 4. 生产示例

无 `DEEPSEEK_API_KEY` 则跳过 LLM；脚本路径仍验证好/坏物料与角色隔离。


In [ ]:
def demo_prod_reviewer() -> None:
    """启发式对照 +（有 key 时）同模型不同角色的审查者。"""
    reset_review_bundle(quality="good")
    heuristic_good = HeuristicReviewer().score(_loop().bundle)
    assert heuristic_good.verdict == "pass"

    reset_review_bundle(quality="bad")
    before = _loop().diff_path.read_text(encoding="utf-8")
    heuristic_bad = ReviewerLoop(_loop().root).review()
    assert heuristic_bad.verdict == "hard_fail"
    assert _loop().diff_path.read_text(encoding="utf-8") == before
    print("=== scripted ===")
    print(json.dumps({"good_pass": True, "bad_hard_fail": True, "diff_untouched": True}, ensure_ascii=False))

    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP llm agent: DEEPSEEK_API_KEY missing")
        print("PROD DEMO OK (scripted only)")
        return

    agent = build_reviewer_agent()
    prompt = (
        "reset_bundle quality=weak，read_checklist，read_goal，read_diff，read_state，"
        "read_feedback，read_verification，按五维打分 write_review_report，"
        "再 try_patch_diff（应失败），read_review_report，"
        "中文说明构建者与审查者为何必须角色分离。"
    )
    result = agent.invoke({"messages": [HumanMessage(content=prompt)]})
    print("=== control agent ===")
    _print_trace(list(result["messages"]))

    report = json.loads(_loop().report_path.read_text(encoding="utf-8"))
    assert report["role"] == "reviewer"
    assert report.get("edited_diff") is False
    assert set(s["dimension"] for s in report["scores"]) == set(DIMENSIONS)
    assert all(0 <= int(s["score"]) <= 2 for s in report["scores"])
    assert report["verdict"] == verdict_from_total(int(report["total"]))
    assert _loop().diff_path.read_text(encoding="utf-8") == _loop().bundle.root.joinpath("diff.json").read_text(encoding="utf-8")
    print("PROD DEMO OK")


demo_prod_reviewer()
